# Reporte 6 – CF 453B Little Pony and Harmony Chest
Código (C++ top-down) y experimentos. Ejecutar las celdas en orden.

In [ ]:
%%writefile solucion.cpp
#include <bits/stdc++.h>
using namespace std;

const int VMAX = 58;          // cota superior para los b_i (ver Lema 1)
const int NP   = 16;          // primos <= 53
int primos[NP] = {2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53};
int n, a[105], msk[VMAX + 1]; // msk[v] = conjunto de primos que dividen a v
int memo[105][1 << NP];       // memo[i][S] = K(i, S), -1 = no calculado
long long visitados = 0;

// K(i, S): costo minimo de elegir b_i..b_n si los primos de S ya estan usados
int K(int i, int S) {
    if (i == n + 1) return 0;                       // caso base
    int &r = memo[i][S];
    if (r != -1) return r;                          // memoizacion
    visitados++;
    r = INT_MAX;
    for (int v = 1; v <= VMAX; v++)                 // decision: valor de b_i
        if ((msk[v] & S) == 0)                      // alternativa factible
            r = min(r, abs(a[i] - v) + K(i + 1, S | msk[v]));
    return r;
}

int main() {
    for (int v = 1; v <= VMAX; v++)
        for (int j = 0; j < NP; j++)
            if (v % primos[j] == 0) msk[v] |= 1 << j;
    scanf("%d", &n);
    for (int i = 1; i <= n; i++) scanf("%d", &a[i]);
    memset(memo, -1, sizeof(memo));
    auto t0 = chrono::steady_clock::now();
    int opt = K(1, 0);
    double ms = chrono::duration<double, milli>(chrono::steady_clock::now() - t0).count();
    // reconstruccion de un b optimo
    int S = 0;
    for (int i = 1; i <= n; i++)
        for (int v = 1; v <= VMAX; v++)
            if ((msk[v] & S) == 0 && abs(a[i] - v) + K(i + 1, S | msk[v]) == K(i, S)) {
                printf("%d ", v); S |= msk[v]; break;
            }
    printf("\n");
    fprintf(stderr, "costo=%d estados=%lld ms=%.2f\n", opt, visitados, ms);
}


In [ ]:
!g++ -O2 -o sol solucion.cpp

In [ ]:
!printf '5\n1 1 1 1 1\n' | ./sol
!printf '5\n1 6 4 2 8\n' | ./sol

In [ ]:
%%writefile experimentos.py
import subprocess, random, re, math, itertools, time
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

def run(a):
    inp = f"{len(a)}\n{' '.join(map(str,a))}\n"
    p = subprocess.run(["./sol"], input=inp, capture_output=True, text=True)
    b = list(map(int, p.stdout.split()))
    m = re.search(r"costo=(\d+) estados=(\d+) ms=([\d.]+)", p.stderr)
    return b, int(m[1]), int(m[2]), float(m[3])

def harmony(b): return all(math.gcd(x,y)==1 for x,y in itertools.combinations(b,2))
def cost(a,b): return sum(abs(x-y) for x,y in zip(a,b))

def brute(a, vmax):
    best=[10**9]
    def rec(i, cur, c):
        if c>=best[0]: return
        if i==len(a): best[0]=c; return
        for v in range(1,vmax+1):
            if all(math.gcd(v,u)==1 for u in cur):
                rec(i+1, cur+[v], c+abs(a[i]-v))
    rec(0,[],0); return best[0]

print("=== PRUEBAS ===")
casos = {
 "Ej.1 (todos 1)":[1]*5, "Ej.2":[1,6,4,2,8], "n=1, a=30":[30], "n=1, a=1":[1],
 "n=2, a=[4,2]":[4,2], "n=3, a=[30,30,30]":[30,30,30], "n=2 iguales [6,6]":[6,6],
 "n=100 todos 1":[1]*100, "n=100 todos 30":[30]*100,
}
for k,a in casos.items():
    b,c,s,ms = run(a)
    assert harmony(b) and cost(a,b)==c
    print(k, "| b =", (b if len(b)<=8 else str(b[:10])+"..."), "| costo =", c, "| harmony:", harmony(b))
b,c,_,_=run([30]*100); print("n=100 todos 30: no-1 en b:", [x for x in b if x!=1], "cantidad:", sum(x!=1 for x in b))

print("=== CONTRASTE CON FUERZA BRUTA ===")
random.seed(7); ok=0; tot=0
for n in (1,2,3):
    for _ in range(40):
        a=[random.randint(1,30) for _ in range(n)]
        _,c,_,_=run(a); assert c==brute(a,100),(a); tot+=1
for _ in range(25):
    a=[random.randint(1,30) for _ in range(4)]
    _,c,_,_=run(a); assert c==brute(a,60),(a); tot+=1
print("casos aleatorios iguales a fuerza bruta:", tot, "de", tot)

print("=== TIEMPOS ===")
ns=list(range(5,101,5)); res=[]
for n in ns:
    ts=[]
    for r in range(7):
        a=[random.randint(1,30) for _ in range(n)]
        _,c,s,ms=run(a); ts.append(ms)
    ts.sort(); res.append((n,ts[3],s)); print(n, "ms=%.1f"%ts[3], "estados=",s)
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot([r[0] for r in res],[r[1] for r in res],"o-"); ax[0].set_xlabel("n (tamaño de la entrada)"); ax[0].set_ylabel("tiempo (ms)"); ax[0].set_title("Tiempo de ejecución vs n"); ax[0].grid(alpha=.3)
ax[1].plot([r[0] for r in res],[r[2] for r in res],"s-",color="C1"); ax[1].set_xlabel("n"); ax[1].set_ylabel("estados (i,S) visitados"); ax[1].set_title("Subproblemas resueltos vs n"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("tiempos.png",dpi=150)
# ajuste lineal t = c*n
import numpy as np
x=np.array([r[0] for r in res]); y=np.array([r[1] for r in res]); 
A=np.vstack([x,np.ones_like(x)]).T; m,c0=np.linalg.lstsq(A,y,rcond=None)[0]
print("ajuste t ~ %.3f*n + %.2f ; R2=%.4f"%(m,c0,1-((y-(m*x+c0))**2).sum()/((y-y.mean())**2).sum()))


In [ ]:
!python experimentos.py

In [ ]:
from IPython.display import Image
Image('tiempos.png')